# INFY Custom Data Source (DB-backed)
This notebook loads INFY daily bars from Postgres via a custom QuantConnect `PythonData` source.

In [63]:
# QuantBook Analysis Tool
qb = QuantBook()

# Force reload so notebook always uses latest db_tick_custom_data.py on disk
import importlib
import db_tick_custom_data
importlib.reload(db_tick_custom_data)
print("db_tick_custom_data loaded from:", db_tick_custom_data.__file__)

DbDailyByTradingsymbol = db_tick_custom_data.DbDailyByTradingsymbol

original_log = DbDailyByTradingsymbol._log.__func__
@classmethod
def quiet_cache_hit_log(cls, message: str) -> None:
    if str(message).startswith("Using existing cache for "):
        return
    return original_log(cls, message)

DbDailyByTradingsymbol._log = quiet_cache_hit_log

db_tick_custom_data loaded from: /Lean/Launcher/bin/Debug/Notebooks/db_tick_custom_data.py


In [64]:
# Request INFY data through custom data subscription
qb.set_start_date(2026, 4, 22)
infy_custom = qb.add_data(DbDailyByTradingsymbol, "INFY", Resolution.DAILY)

# Get 360 daily bars, similar to Cell 2 style
history = qb.history(infy_custom.symbol, 360, Resolution.DAILY)
history.tail()

[DbDailyByTradingsymbol] Refreshing cache for INFY
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=INFY
[DbDailyByTradingsymbol] Resolved instrument_token=408065
[DbDailyByTradingsymbol] Fetched 5745 rows from daily_candles, date range [2003-01-01 -> 2026-04-22]
[DbDailyByTradingsymbol] Latest row close/volume = 1257.9/10635696 on 2026-04-22
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/INFY.csv


close    high     low    open  \
symbol                      time                                         
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  1331.0  1309.0  1322.1   
                            2026-04-17  1318.7  1328.3  1306.1  1313.0   
                            2026-04-20  1312.6  1324.9  1308.1  1320.0   
                            2026-04-21  1313.2  1325.0  1299.3  1310.0   
                            2026-04-22  1257.9  1297.7  1257.0  1295.0   

                                         value      volume  
symbol                      time                            
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  18527029.0  
                            2026-04-17  1318.7  12064048.0  
                            2026-04-20  1312.6   7172300.0  
                            2026-04-21  1313.2  10057440.0  
                            2026-04-22  1257.9  10635696.0

In [49]:
# Direct DB check from the same SessionLocal used by custom source
from sqlalchemy import text, select
from db_tick_custom_data import SessionLocal, Instrument

with SessionLocal() as s:
    token = s.execute(
        select(Instrument.instrument_token).where(
            Instrument.tradingsymbol == "INFY",
            Instrument.exchange == "NSE",
            Instrument.active.is_(True),
        )
    ).scalar_one()
    rows = s.execute(
        text("""
            SELECT candle_date, open, high, low, close, volume
            FROM daily_candles
            WHERE instrument_token = :t
            ORDER BY candle_date DESC
            LIMIT 5
        """),
        {"t": token},
    ).all()

print("instrument_token:", token)
for r in rows:
    print(r)

instrument_token: 408065
(datetime.date(2026, 4, 22), 1295.0, 1297.7, 1257.0, 1257.9, 10635696)
(datetime.date(2026, 4, 21), 1310.0, 1325.0, 1299.3, 1313.2, 10057440)
(datetime.date(2026, 4, 20), 1320.0, 1324.9, 1308.1, 1312.6, 7172300)
(datetime.date(2026, 4, 17), 1313.0, 1328.3, 1306.1, 1318.7, 12064048)
(datetime.date(2026, 4, 16), 1322.1, 1331.0, 1309.0, 1319.2, 18527029)
